In [2]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

df = pd.read_csv('./data/application_train.csv')
print(df.shape)

(307511, 122)


In [3]:
# Corrections
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df = df[df['CODE_GENDER'] != 'XNA']

# Features dérivées
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['EMPLOYED_TO_AGE_RATIO'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

# Drop colonnes +40% missing sauf EXT_SOURCE_1
missing_pct = df.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 40].index.tolist()
cols_to_drop = [c for c in cols_to_drop if c != 'EXT_SOURCE_1']
df = df.drop(columns=cols_to_drop)

# Drop colonnes non pertinentes
df = df.drop(columns=['SK_ID_CURR', 'WEEKDAY_APPR_PROCESS_START'])

X = df.drop(columns=['TARGET'])
y = df['TARGET']

print(X.shape)

(307507, 76)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
import xgboost as xgb

# Identifier les colonnes
num_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# Pipeline numérique — imputation médiane + winsorization
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

# Pipeline catégoriel — imputation + one-hot encoding
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combiner
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

print("Preprocessor défini ✓")

Preprocessor défini ✓


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBClassifier(
        n_estimators=494,
        max_depth=4,
        learning_rate=0.086,
        subsample=0.786,
        colsample_bytree=0.932,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='auc'
    ))
])

full_pipeline.fit(X_train, y_train)
print("Pipeline entraîné ✓")

Pipeline entraîné ✓


In [6]:
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp

y_pred = full_pipeline.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_pred)
gini = 2 * auc - 1
ks, _ = ks_2samp(y_pred[y_test == 1], y_pred[y_test == 0])

print(f"AUC-ROC : {auc:.4f}")
print(f"Gini    : {gini:.4f}")
print(f"KS      : {ks:.4f}")

AUC-ROC : 0.7703
Gini    : 0.5405
KS      : 0.4024


In [7]:
import pickle

with open('pipeline.pkl', 'wb') as f:
    pickle.dump(full_pipeline, f)

print("Pipeline sauvegardé ✓")

Pipeline sauvegardé ✓


In [9]:
test = pd.read_csv('./data/application_test.csv')
test_ids = test['SK_ID_CURR']

# Mêmes corrections
test['DAYS_EMPLOYED'] = test['DAYS_EMPLOYED'].replace(365243, np.nan)
test = test.drop(columns=cols_to_drop, errors='ignore')
test = test.drop(columns=['SK_ID_CURR', 'WEEKDAY_APPR_PROCESS_START'], errors='ignore')

# Features dérivées
test['CREDIT_INCOME_RATIO'] = test['AMT_CREDIT'] / test['AMT_INCOME_TOTAL']
test['ANNUITY_INCOME_RATIO'] = test['AMT_ANNUITY'] / test['AMT_INCOME_TOTAL']
test['EMPLOYED_TO_AGE_RATIO'] = test['DAYS_EMPLOYED'] / test['DAYS_BIRTH']
test['CREDIT_TERM'] = test['AMT_ANNUITY'] / test['AMT_CREDIT']
test['INCOME_PER_PERSON'] = test['AMT_INCOME_TOTAL'] / test['CNT_FAM_MEMBERS']

# Le Pipeline gère le reste automatiquement
y_pred_test = full_pipeline.predict_proba(test)[:, 1]

submission = pd.DataFrame({
    'SK_ID_CURR': test_ids,
    'TARGET': y_pred_test
})

submission.to_csv('data/clean/submission_pipeline.csv', index=False)
print(submission.head())

   SK_ID_CURR    TARGET
0      100001  0.186937
1      100005  0.600514
2      100013  0.075707
3      100028  0.312109
4      100038  0.622293


In [11]:
bureau = pd.read_csv('./data/bureau.csv')
print(bureau.shape)
print(bureau.head())
print(bureau.dtypes)

(1716428, 17)
   SK_ID_CURR  SK_ID_BUREAU CREDIT_ACTIVE CREDIT_CURRENCY  DAYS_CREDIT  \
0      215354       5714462        Closed      currency 1         -497   
1      215354       5714463        Active      currency 1         -208   
2      215354       5714464        Active      currency 1         -203   
3      215354       5714465        Active      currency 1         -203   
4      215354       5714466        Active      currency 1         -629   

   CREDIT_DAY_OVERDUE  DAYS_CREDIT_ENDDATE  DAYS_ENDDATE_FACT  \
0                   0               -153.0             -153.0   
1                   0               1075.0                NaN   
2                   0                528.0                NaN   
3                   0                  NaN                NaN   
4                   0               1197.0                NaN   

   AMT_CREDIT_MAX_OVERDUE  CNT_CREDIT_PROLONG  AMT_CREDIT_SUM  \
0                     NaN                   0         91323.0   
1                   

In [12]:
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_LOAN_COUNT=('SK_ID_BUREAU', 'count'),
    BUREAU_ACTIVE_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    BUREAU_CLOSED_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),
    BUREAU_BAD_DEBT_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Bad debt').sum()),
    BUREAU_MAX_OVERDUE=('CREDIT_DAY_OVERDUE', 'max'),
    BUREAU_TOTAL_DEBT=('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUREAU_TOTAL_CREDIT=('AMT_CREDIT_SUM', 'sum'),
    BUREAU_AVG_DAYS_CREDIT=('DAYS_CREDIT', 'mean'),
).reset_index()

bureau_agg['BUREAU_ACTIVE_RATIO'] = bureau_agg['BUREAU_ACTIVE_COUNT'] / bureau_agg['BUREAU_LOAN_COUNT']

print(bureau_agg.shape)
print(bureau_agg.head())

(305811, 10)
   SK_ID_CURR  BUREAU_LOAN_COUNT  BUREAU_ACTIVE_COUNT  BUREAU_CLOSED_COUNT  \
0      100001                  7                    3                    4   
1      100002                  8                    2                    6   
2      100003                  4                    1                    3   
3      100004                  2                    0                    2   
4      100005                  3                    2                    1   

   BUREAU_BAD_DEBT_COUNT  BUREAU_MAX_OVERDUE  BUREAU_TOTAL_DEBT  \
0                      0                   0           596686.5   
1                      0                   0           245781.0   
2                      0                   0                0.0   
3                      0                   0                0.0   
4                      0                   0           568408.5   

   BUREAU_TOTAL_CREDIT  BUREAU_AVG_DAYS_CREDIT  BUREAU_ACTIVE_RATIO  
0          1453365.000             -735.00000

In [14]:
# Merger avec le dataset principal
df = pd.read_csv('./data/application_train.csv')

# Mêmes corrections qu'avant
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df = df[df['CODE_GENDER'] != 'XNA']

# Features dérivées
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['EMPLOYED_TO_AGE_RATIO'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

# Drop colonnes +40% missing
missing_pct = df.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 40].index.tolist()
cols_to_drop = [c for c in cols_to_drop if c != 'EXT_SOURCE_1']
df = df.drop(columns=cols_to_drop)
df = df.drop(columns=['WEEKDAY_APPR_PROCESS_START'])

# Merger bureau
df = df.merge(bureau_agg, on='SK_ID_CURR', how='left')

# Clients sans historique bureau → 0
bureau_cols = bureau_agg.columns.tolist()
bureau_cols.remove('SK_ID_CURR')
df[bureau_cols] = df[bureau_cols].fillna(0)

X = df.drop(columns=['SK_ID_CURR', 'TARGET'])
y = df['TARGET']

print(X.shape)

(307507, 85)


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

num_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

full_pipeline_v2 = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ])),
    ('model', xgb.XGBClassifier(
        n_estimators=494,
        max_depth=4,
        learning_rate=0.086,
        subsample=0.786,
        colsample_bytree=0.932,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='auc'
    ))
])

full_pipeline_v2.fit(X_train, y_train)

y_pred = full_pipeline_v2.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred)
gini = 2 * auc - 1
ks, _ = ks_2samp(y_pred[y_test == 1], y_pred[y_test == 0])

print(f"AUC-ROC : {auc:.4f}")
print(f"Gini    : {gini:.4f}")
print(f"KS      : {ks:.4f}")

AUC-ROC : 0.7717
Gini    : 0.5434
KS      : 0.4050


In [17]:
test = pd.read_csv('./data/application_test.csv')
test_ids = test['SK_ID_CURR']

# Mêmes corrections
test['DAYS_EMPLOYED'] = test['DAYS_EMPLOYED'].replace(365243, np.nan)
test = test.drop(columns=cols_to_drop, errors='ignore')
test = test.drop(columns=['WEEKDAY_APPR_PROCESS_START'], errors='ignore')

# Features dérivées
test['CREDIT_INCOME_RATIO'] = test['AMT_CREDIT'] / test['AMT_INCOME_TOTAL']
test['ANNUITY_INCOME_RATIO'] = test['AMT_ANNUITY'] / test['AMT_INCOME_TOTAL']
test['EMPLOYED_TO_AGE_RATIO'] = test['DAYS_EMPLOYED'] / test['DAYS_BIRTH']
test['CREDIT_TERM'] = test['AMT_ANNUITY'] / test['AMT_CREDIT']
test['INCOME_PER_PERSON'] = test['AMT_INCOME_TOTAL'] / test['CNT_FAM_MEMBERS']

# Merger bureau
test = test.merge(bureau_agg, on='SK_ID_CURR', how='left')
test[bureau_cols] = test[bureau_cols].fillna(0)
test = test.drop(columns=['SK_ID_CURR'], errors='ignore')

y_pred_test = full_pipeline_v2.predict_proba(test)[:, 1]

submission = pd.DataFrame({'SK_ID_CURR': test_ids, 'TARGET': y_pred_test})
submission.to_csv('data/clean/submission_bureau.csv', index=False)
print(submission.head())

   SK_ID_CURR    TARGET
0      100001  0.142150
1      100005  0.558072
2      100013  0.068314
3      100028  0.279177
4      100038  0.616352


In [19]:
prev = pd.read_csv('./data/previous_application.csv')
print(prev.shape)
print(prev.columns.tolist())

(1670214, 37)
['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'FLAG_LAST_APPL_PER_CONTRACT', 'NFLAG_LAST_APPL_IN_DAY', 'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED', 'NAME_CASH_LOAN_PURPOSE', 'NAME_CONTRACT_STATUS', 'DAYS_DECISION', 'NAME_PAYMENT_TYPE', 'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE', 'SELLERPLACE_AREA', 'NAME_SELLER_INDUSTRY', 'CNT_PAYMENT', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION', 'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION', 'NFLAG_INSURED_ON_APPROVAL']


In [21]:
prev = pd.read_csv('./data/previous_application.csv')

prev_agg = prev.groupby('SK_ID_CURR').agg(
    PREV_APP_COUNT=('SK_ID_PREV', 'count'),
    PREV_APPROVED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    PREV_REFUSED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    PREV_AMT_CREDIT_MEAN=('AMT_CREDIT', 'mean'),
    PREV_AMT_ANNUITY_MEAN=('AMT_ANNUITY', 'mean'),
    PREV_DAYS_DECISION_MEAN=('DAYS_DECISION', 'mean'),
    PREV_CNT_PAYMENT_MEAN=('CNT_PAYMENT', 'mean'),
).reset_index()

prev_agg['PREV_REFUSAL_RATE'] = prev_agg['PREV_REFUSED_COUNT'] / prev_agg['PREV_APP_COUNT']

print(prev_agg.shape)
print(prev_agg.head())

(338857, 9)
   SK_ID_CURR  PREV_APP_COUNT  PREV_APPROVED_COUNT  PREV_REFUSED_COUNT  \
0      100001               1                    1                   0   
1      100002               1                    1                   0   
2      100003               3                    3                   0   
3      100004               1                    1                   0   
4      100005               2                    1                   0   

   PREV_AMT_CREDIT_MEAN  PREV_AMT_ANNUITY_MEAN  PREV_DAYS_DECISION_MEAN  \
0              23787.00               3951.000                  -1740.0   
1             179055.00               9251.775                   -606.0   
2             484191.00              56553.990                  -1305.0   
3              20106.00               5357.250                   -815.0   
4              20076.75               4813.200                   -536.0   

   PREV_CNT_PAYMENT_MEAN  PREV_REFUSAL_RATE  
0                    8.0                0.0  


In [22]:
df = pd.read_csv('./data/application_train.csv')

# Mêmes corrections
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df = df[df['CODE_GENDER'] != 'XNA']

# Features dérivées
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['EMPLOYED_TO_AGE_RATIO'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

# Drop colonnes
missing_pct = df.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 40].index.tolist()
cols_to_drop = [c for c in cols_to_drop if c != 'EXT_SOURCE_1']
df = df.drop(columns=cols_to_drop)
df = df.drop(columns=['WEEKDAY_APPR_PROCESS_START'])

# Merger bureau + previous
df = df.merge(bureau_agg, on='SK_ID_CURR', how='left')
df = df.merge(prev_agg, on='SK_ID_CURR', how='left')

# Fillna pour clients sans historique
for col in bureau_cols:
    df[col] = df[col].fillna(0)

prev_cols = prev_agg.columns.tolist()
prev_cols.remove('SK_ID_CURR')
for col in prev_cols:
    df[col] = df[col].fillna(0)

X = df.drop(columns=['SK_ID_CURR', 'TARGET'])
y = df['TARGET']

print(X.shape)

(307507, 93)


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

num_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

full_pipeline_v3 = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ])),
    ('model', xgb.XGBClassifier(
        n_estimators=494,
        max_depth=4,
        learning_rate=0.086,
        subsample=0.786,
        colsample_bytree=0.932,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='auc'
    ))
])

full_pipeline_v3.fit(X_train, y_train)

y_pred = full_pipeline_v3.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred)
gini = 2 * auc - 1
ks, _ = ks_2samp(y_pred[y_test == 1], y_pred[y_test == 0])

print(f"AUC-ROC : {auc:.4f}")
print(f"Gini    : {gini:.4f}")
print(f"KS      : {ks:.4f}")

AUC-ROC : 0.7760
Gini    : 0.5521
KS      : 0.4095


In [25]:
test = pd.read_csv('./data/application_test.csv')
test_ids = test['SK_ID_CURR']

test['DAYS_EMPLOYED'] = test['DAYS_EMPLOYED'].replace(365243, np.nan)
test = test.drop(columns=cols_to_drop, errors='ignore')
test = test.drop(columns=['WEEKDAY_APPR_PROCESS_START'], errors='ignore')

test['CREDIT_INCOME_RATIO'] = test['AMT_CREDIT'] / test['AMT_INCOME_TOTAL']
test['ANNUITY_INCOME_RATIO'] = test['AMT_ANNUITY'] / test['AMT_INCOME_TOTAL']
test['EMPLOYED_TO_AGE_RATIO'] = test['DAYS_EMPLOYED'] / test['DAYS_BIRTH']
test['CREDIT_TERM'] = test['AMT_ANNUITY'] / test['AMT_CREDIT']
test['INCOME_PER_PERSON'] = test['AMT_INCOME_TOTAL'] / test['CNT_FAM_MEMBERS']

test = test.merge(bureau_agg, on='SK_ID_CURR', how='left')
test = test.merge(prev_agg, on='SK_ID_CURR', how='left')

for col in bureau_cols + prev_cols:
    test[col] = test[col].fillna(0)

test = test.drop(columns=['SK_ID_CURR'], errors='ignore')

y_pred_test = full_pipeline_v3.predict_proba(test)[:, 1]

submission = pd.DataFrame({'SK_ID_CURR': test_ids, 'TARGET': y_pred_test})
submission.to_csv('data/clean/submission_prev.csv', index=False)
print(submission.head())

   SK_ID_CURR    TARGET
0      100001  0.162740
1      100005  0.566336
2      100013  0.095943
3      100028  0.263008
4      100038  0.617210


In [26]:
import optuna

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 600),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'scale_pos_weight': scale_pos_weight,
        'random_state': 42,
        'eval_metric': 'auc'
    }

    pipeline = Pipeline([
        ('preprocessor', ColumnTransformer([
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
                ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
            ]), cat_cols)
        ])),
        ('model', xgb.XGBClassifier(**params))
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict_proba(X_test)[:, 1]
    return roc_auc_score(y_test, y_pred)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"Meilleur AUC : {study.best_value:.4f}")
print(f"Meilleurs params : {study.best_params}")

Meilleur AUC : 0.7776
Meilleurs params : {'n_estimators': 600, 'max_depth': 3, 'learning_rate': 0.10668521333410673, 'subsample': 0.863513373035062, 'colsample_bytree': 0.7806981822398925, 'min_child_weight': 1, 'gamma': 0.5672348525148126}


In [27]:
best_params = study.best_params
best_params['scale_pos_weight'] = scale_pos_weight
best_params['random_state'] = 42
best_params['eval_metric'] = 'auc'

final_pipeline = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ])),
    ('model', xgb.XGBClassifier(**best_params))
])

final_pipeline.fit(X_train, y_train)

y_pred = final_pipeline.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred)
print(f"AUC-ROC final : {auc:.4f}")

# Submission
y_pred_test = final_pipeline.predict_proba(test)[:, 1]
submission = pd.DataFrame({'SK_ID_CURR': test_ids, 'TARGET': y_pred_test})
submission.to_csv('data/clean/submission_final.csv', index=False)
print("Submission sauvegardée ✓")


AUC-ROC final : 0.7776
Submission sauvegardée ✓


In [28]:
with open('pipeline_final.pkl', 'wb') as f:
    pickle.dump(final_pipeline, f)

print("Pipeline final sauvegardé ✓")

Pipeline final sauvegardé ✓


In [29]:
import os
os.makedirs('data/clean', exist_ok=True)

X_train_df = pd.DataFrame(X_train, columns=X.columns)
X_test_df = pd.DataFrame(X_test, columns=X.columns)

X_train_df.to_csv('data/clean/X_train.csv', index=False)
X_test_df.to_csv('data/clean/X_test.csv', index=False)
y_train.to_csv('data/clean/y_train.csv', index=False)
y_test.to_csv('data/clean/y_test.csv', index=False)

print(f"X_train : {X_train_df.shape}")
print(f"X_test : {X_test_df.shape}")

X_train : (246005, 93)
X_test : (61502, 93)
